# Approach E1 – Slide-level mosaic building

**Representation of each whole-slide image by a 4000 × 4000 px mosaic of 16 representative patches selected by *k*-medoids clustering of colorimetric descriptors**

*The mosaics produced here are the input of `02_feature_extraction.ipynb` (KimiaNet).*

## Overview

Approach E1 summarises every slide with a single image. For each slide, the patches extracted by `00_preprocesamiento_propio/wsi_step2_patch_extraction.py` are processed as follows:

1. Patches whose *cell index* (mean optical density) is an outlier within the slide are discarded (Section 1).
2. The colorimetric descriptors of the remaining patches are normalised to unit sum per patch.
3. A pairwise distance matrix is computed with one of four distances: cosine, Euclidean, Jensen–Shannon or Bhattacharyya.
4. *k*-medoids clustering (*k* = 16) is applied to the distance matrix, and the 16 medoid patches are retained.
5. The medoid patches (1000 × 1000 px each) are tiled into a 4 × 4 mosaic of 4000 × 4000 px (Section 3).

Section 2 examines, on the training partition, how the *k*-medoids inertia varies with the number of clusters for each distance.

*The mosaic-building procedure was adapted from the original TCGA-BRCA pipeline of [COLLABORATOR — TO BE COMPLETED].*

## Contents

1. Configuration and helper functions
2. Patch counts after outlier filtering
3. Choice of the number of clusters
4. Mosaic building
5. Notes and limitations

## 1. Configuration and helper functions

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from skimage import io

In [ ]:
# ----------------------------------------------------------------------------
# Repository layout (the notebook is executed from 02_histopatologia/E1_kimianet_mosaico/)
# ----------------------------------------------------------------------------
DATA_DIR = Path("data")     # adjust to the location of the input files

PATCHES_DIR = DATA_DIR / "wsi_processed"                     # output of steps 1-2: one sub-directory per slide
MOSAICS_DIR = DATA_DIR / "mosaics"                           # output of this notebook: <distance>/<slide_id>.png
PARTITION_FILE = DATA_DIR / "wsi_partition_metadata.csv"     # columns: IDs, partition ("Train", ...)

FIGURES_DIR = Path("figures")
TABLES_DIR = Path("results")
OUTPUT_PREFIX = "wsi_e1"
for directory in (MOSAICS_DIR, FIGURES_DIR, TABLES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------------------
# Mosaic parameters
# ----------------------------------------------------------------------------
N_MEDOIDS = 16                # number of representative patches per slide
MOSAIC_GRID = 4               # the mosaic is a MOSAIC_GRID x MOSAIC_GRID array of patches
PATCH_SIZE = 1000             # px, size of the saved patches (JPEG files from step 2)
CLUSTERS_RANGE = (2, 42)      # numbers of clusters explored in Section 3: [start, stop)
RANDOM_STATE = 0
MAX_SLIDES = None             # e.g. 3 for a dry run; None = all slides

# Upper outlier fence for the 'cell index' of the patches of a slide:
#   upper = percentile(UPPER_FENCE_PERCENTILE) + 1.5 * IQR.
# The value 25 reproduces the original pipeline (the fence is anchored at Q1).
# The conventional Tukey fence is anchored at Q3, i.e. 75; see Section 5.
UPPER_FENCE_PERCENTILE = 25

assert N_MEDOIDS == MOSAIC_GRID ** 2, "The mosaic must hold exactly N_MEDOIDS patches."

In [ ]:
def bhattacharyya(x, y):
    """Bhattacharyya distance between two discrete distributions (non-negative, unit-sum vectors)."""
    return -np.log(np.sum(np.sqrt(x * y)))


# Output sub-directory name -> metric accepted by scipy.spatial.distance.pdist
DISTANCES = {
    "cosine": "cosine",
    "euclidean": "euclidean",
    "jensenshannon": "jensenshannon",
    "bhattacharyya": bhattacharyya,
}


def list_slides(patches_dir=PATCHES_DIR):
    """Identifiers of the slides available in ``patches_dir`` (sub-directories named 'TCGA-...')."""
    slides = sorted(p.name for p in Path(patches_dir).iterdir() if p.is_dir() and "TCGA-" in p.name)
    return slides[:MAX_SLIDES]


def load_filtered_patch_info(slide_id, patches_dir=PATCHES_DIR):
    """Read ``patch_info.csv`` of a slide and discard patches with an outlying 'cell index'."""
    patch_info = pd.read_csv(Path(patches_dir) / slide_id / "patch_info.csv")
    cell_index = patch_info["cell index"]
    q1, q3 = np.percentile(cell_index, [25, 75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = np.percentile(cell_index, UPPER_FENCE_PERCENTILE) + 1.5 * iqr
    return patch_info[(cell_index > lower) & (cell_index < upper)].reset_index(drop=True)


def descriptor_matrix(patch_info):
    """Descriptors of every patch (all columns after 'ID' and 'cell index'), normalised to unit sum per patch."""
    features = patch_info[patch_info.columns[2:]].to_numpy(dtype=np.float64)
    row_sums = features.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1          # avoid division by zero
    return features / row_sums

In [ ]:
class KMedoids:
    """Alternating (Voronoi-iteration) k-medoids on a precomputed distance matrix.

    The medoids are initialised at random (seeded) and then refined by alternating
    (i) the assignment of every point to its nearest medoid and (ii) the replacement
    of each medoid by the member of its cluster with the smallest summed distance to
    the other members, until no medoid changes. This is not the PAM algorithm (there
    is no swap phase). Interface: ``fit(D)`` sets ``medoid_indices_``, ``labels_`` and
    ``inertia_`` (sum of the distances of all points to their closest medoid).
    """

    def __init__(self, n_clusters=16, random_state=0, max_iter=300):
        self.n_clusters = n_clusters
        self.random_state = random_state
        self.max_iter = max_iter

    def fit(self, D):
        rng = np.random.RandomState(self.random_state)
        n = D.shape[0]
        medoids = rng.choice(n, self.n_clusters, replace=False)

        for _ in range(self.max_iter):
            labels = np.argmin(D[:, medoids], axis=1)
            new_medoids = medoids.copy()
            changed = False
            for k in range(self.n_clusters):
                members = np.where(labels == k)[0]
                if len(members) == 0:
                    continue
                within = D[np.ix_(members, members)]
                best = members[np.argmin(within.sum(axis=1))]
                if best != new_medoids[k]:
                    new_medoids[k] = best
                    changed = True
            medoids = new_medoids
            if not changed:
                break

        self.medoid_indices_ = medoids
        self.labels_ = np.argmin(D[:, medoids], axis=1)
        self.inertia_ = float(D[np.arange(n), medoids[self.labels_]].sum())
        return self

In [ ]:
def build_mosaic(slide_id, out_path, metric, patches_dir=PATCHES_DIR):
    """Build and save the mosaic of one slide. Returns a status string.

    Statuses: 'built', 'exists' (already generated) or 'too_few_patches' (fewer than
    N_MEDOIDS patches remain after the outlier filter).
    """
    if out_path.exists():
        return "exists"

    patch_info = load_filtered_patch_info(slide_id, patches_dir)
    if len(patch_info) < N_MEDOIDS:
        return "too_few_patches"

    distances = squareform(pdist(descriptor_matrix(patch_info), metric=metric))
    medoids = KMedoids(n_clusters=N_MEDOIDS, random_state=RANDOM_STATE).fit(distances).medoid_indices_
    medoid_ids = patch_info["ID"].to_numpy()[medoids]

    side = MOSAIC_GRID * PATCH_SIZE
    mosaic = np.empty([side, side, 3], dtype=np.uint8)
    row = col = 0                          # patches fill the mosaic column by column
    for patch_id in medoid_ids:
        patch = io.imread(Path(patches_dir) / slide_id / f"{int(patch_id)}.jpg")
        mosaic[row * PATCH_SIZE:(row + 1) * PATCH_SIZE, col * PATCH_SIZE:(col + 1) * PATCH_SIZE, :] = patch
        row += 1
        if row == MOSAIC_GRID:
            row, col = 0, col + 1

    io.imsave(out_path, mosaic, check_contrast=False)
    return "built"

## 2. Patch counts after outlier filtering

Number of patches retained per slide after discarding those with an outlying *cell index*. Slides with fewer than `N_MEDOIDS` retained patches cannot be summarised and are skipped in Section 4.

In [ ]:
slides = list_slides()
print(f"Slides found: {len(slides)}")

patch_counts = pd.Series({slide: len(load_filtered_patch_info(slide)) for slide in slides}, name="patches_after_filter")

plt.figure(figsize=(4, 5))
plt.boxplot(patch_counts.values)
plt.ylabel("Patches per slide (after filtering)")
plt.xticks([])
plt.title("Patch counts")
plt.savefig(FIGURES_DIR / f"{OUTPUT_PREFIX}_patch_counts_boxplot.png")
plt.show()

print(patch_counts.describe().round(1).to_string())
print(f"\nSlides with fewer than {N_MEDOIDS} patches: {int((patch_counts < N_MEDOIDS).sum())}")

## 3. Choice of the number of clusters

For every slide of the training partition and every distance, *k*-medoids is fitted for *k* = 2, …, 41, and the inertia is min–max normalised within each slide. The mean ± SD across slides is plotted against *k*. The value of *k* reported in each panel title is the point of the mean curve closest to the origin once *k* is rescaled to [0, 1], a simple criterion for the elbow of the curve.

The partition file must contain the columns `IDs` and `partition` (`Train` for the training slides). If the file is not available, this section is skipped.

In [ ]:
if not PARTITION_FILE.exists():
    print(f"Partition file not found ({PARTITION_FILE}); Section 3 skipped.")
else:
    partition = pd.read_csv(PARTITION_FILE)
    train_ids = partition.loc[partition["partition"] == "Train", "IDs"].to_list()[:MAX_SLIDES]
    k_values = np.arange(*CLUSTERS_RANGE)

    # Slides need more patches than the largest k explored
    eligible = [s for s in train_ids if len(load_filtered_patch_info(s)) > k_values.max()]
    print(f"Training slides: {len(train_ids)} | eligible (more than {k_values.max()} patches): {len(eligible)}")
    train_ids = eligible

    inertia = np.zeros([len(train_ids), len(k_values), len(DISTANCES)])
    for d, (dist_name, metric) in enumerate(DISTANCES.items()):
        for s, slide_id in enumerate(train_ids):
            patch_info = load_filtered_patch_info(slide_id)
            distances = squareform(pdist(descriptor_matrix(patch_info), metric=metric))
            values = np.array([KMedoids(n_clusters=int(k), random_state=RANDOM_STATE).fit(distances).inertia_
                               for k in k_values])
            inertia[s, :, d] = (values - values.min()) / (values.max() - values.min())

    fig, axes = plt.subplots(1, len(DISTANCES), figsize=(20, 5), tight_layout=True)
    best_k = {}
    for d, dist_name in enumerate(DISTANCES):
        mean, sd = inertia[:, :, d].mean(axis=0), inertia[:, :, d].std(axis=0)
        k_scaled = (k_values - k_values.min()) / (k_values.max() - k_values.min())
        best_k[dist_name] = int(k_values[np.argmin(mean ** 2 + k_scaled ** 2)])

        axes[d].plot(k_values, mean)
        axes[d].fill_between(k_values, mean - sd, mean + sd, color="skyblue", alpha=0.4)
        axes[d].set(title=f"{dist_name}: k = {best_k[dist_name]}", xlabel="Number of clusters (k)",
                    ylabel="Normalised inertia")
    plt.savefig(FIGURES_DIR / f"{OUTPUT_PREFIX}_kmedoids_inertia_vs_k.png")
    plt.show()

    best_k_table = pd.Series(best_k, name="k_elbow").rename_axis("distance").to_frame()
    best_k_table.to_csv(TABLES_DIR / f"{OUTPUT_PREFIX}_elbow_k_by_distance.csv")
    print(best_k_table.to_string())

## 4. Mosaic building

A mosaic is generated for every slide and every distance and saved as `MOSAICS_DIR/<distance>/<slide_id>.png`. Existing mosaics are not regenerated, so the cell can be re-run after an interruption. A build log is stored in `results/`.

### 4.1 Identifier check

TCGA patient barcodes have 12 characters (`TCGA-XX-XXXX`). Slide identifiers that are longer, or that share their first 12 characters with another slide, may correspond to duplicated patients and should be reviewed before the features are merged with the clinical and transcriptomic tables.

In [ ]:
patient_of_slide = pd.Series({slide: slide[:12] for slide in slides}, name="patient_id")
long_ids = patient_of_slide[[len(slide) > 12 for slide in patient_of_slide.index]]
duplicated = patient_of_slide[patient_of_slide.duplicated(keep=False)]

print(f"Identifiers longer than 12 characters: {len(long_ids)}")
print(long_ids.to_string() if len(long_ids) else "  none")
print(f"\nSlides sharing a 12-character patient identifier: {len(duplicated)}")
print(duplicated.sort_values().to_string() if len(duplicated) else "  none")

### 4.2 Generation

In [ ]:
records = []
for dist_name, metric in DISTANCES.items():
    out_dir = MOSAICS_DIR / dist_name
    out_dir.mkdir(parents=True, exist_ok=True)
    for slide_id in slides:
        try:
            status = build_mosaic(slide_id, out_dir / f"{slide_id}.png", metric)
        except Exception as exc:                       # keep going; the error is logged
            status = f"error: {type(exc).__name__}: {exc}"
        records.append({"distance": dist_name, "slide_id": slide_id, "status": status})

mosaic_log = pd.DataFrame(records)
mosaic_log.to_csv(TABLES_DIR / f"{OUTPUT_PREFIX}_mosaic_build_log.csv", index=False)
mosaic_log.groupby(["distance", "status"]).size().unstack(fill_value=0)

## 5. Notes and limitations

1. **Outlier fence.** The upper fence is anchored at the first quartile (`UPPER_FENCE_PERCENTILE = 25`), whereas the conventional Tukey fence is `Q3 + 1.5·IQR`. The value is kept for consistency with the mosaics of the slides processed by the original pipeline; the effect of the conventional fence can be assessed by setting the parameter to 75.
2. **Clustering algorithm.** The implementation is an alternating *k*-medoids with seeded random initialisation (see the class docstring). It replaces `sklearn_extra.cluster.KMedoids`, which could not be imported in the working environment because of a NumPy binary incompatibility. Results may differ from those of `sklearn_extra`, whose default initialisation is not random.
3. **Descriptors.** The descriptor vector comprises all columns after `ID` and `cell index` in `patch_info.csv`, i.e. the 14 colorimetric descriptors and the tissue coverage of the patch. Each patch vector is normalised to unit sum, which treats heterogeneous descriptors (means, standard deviations, hue, optical density) as a distribution.
4. **Heterogeneous preprocessing.** Slides preprocessed with the scripts of `00_preprocesamiento_propio` and slides processed by the original pipeline may differ in the exact definition of the colorimetric descriptors and of the *cell index*; a systematic difference between the two groups (batch effect) should be ruled out before pooling them.